# Green Roof Random Forest — Training and Evaluation

This notebook provides a compact, publication-oriented version of the Random Forest workflow used for green-roof classification.

The model combines:

- LoD2 roof-plane geometry and semantic attributes
- seasonal remote-sensing statistics
- NDVI summary statistics
- a scikit-learn preprocessing pipeline
- a Random Forest classifier

## Workflow

1. Configure project-relative paths
2. Load and quality-filter the prepared training dataset
3. Define the final feature set
4. Build the preprocessing + Random Forest pipeline
5. Evaluate geographic generalisation with Leave-One-City-Out (LOCO) cross-validation
6. Compare with repeated stratified mixed-sample cross-validation
7. Evaluate a validation-selected threshold on a held-out test split
8. Train the final model on all labelled data
9. Export the fitted model
10. Run a small inference example

> **Data availability:** Raw and intermediate geospatial datasets are not included in the public repository because of file size and/or licensing constraints. See `data/README.md` for the expected input schema.


## 1. Imports and project configuration


In [ ]:
from pathlib import Path
import sys
import time

import joblib
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.model_selection import (
    LeaveOneGroupOut,
    RepeatedStratifiedKFold,
    train_test_split,
)
from sklearn.metrics import (
    average_precision_score,
    precision_recall_curve,
    balanced_accuracy_score,
    accuracy_score,
    f1_score,
)

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

print("Python:", sys.version.split()[0])
print("GeoPandas:", gpd.__version__)
print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)


In [ ]:
# This notebook is intended to live in: <repo>/notebooks/
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

DATA_DIR = PROJECT_ROOT / "data"
MODEL_DIR = PROJECT_ROOT / "models"

TRAIN_PATH = DATA_DIR / "training_roofs.parquet"
MODEL_PATH = MODEL_DIR / "green_roof_rf_v1.0.0.joblib"

MODEL_DIR.mkdir(parents=True, exist_ok=True)

ALLOWED_METHODS = {1, 2}
TARGET_COL = "label"
GROUP_COL = "city"

print("Project root:", PROJECT_ROOT)
print("Training data:", TRAIN_PATH)
print("Model output:", MODEL_PATH)


## 2. Final feature set

The final classifier uses 14 roof-level features: 12 numeric variables and 2 categorical attributes.


In [ ]:
FEATURES_FINAL = [
    "slope_deg",
    "area_m2",
    "height_m",
    "height_relief_m",
    "G_avg_Winter",
    "B_avg_Winter",
    "NDVI_avg_Winter",
    "NDVI_std_Winter",
    "G_avg_Summer",
    "B_avg_Summer",
    "NDVI_avg_Summer",
    "NDVI_std_Summer",
    "roofType",
    "function",
]

CAT_COLS = ["roofType", "function"]
NUM_COLS = [c for c in FEATURES_FINAL if c not in CAT_COLS]

print("Final feature count:", len(FEATURES_FINAL))
print("Numeric:", len(NUM_COLS))
print("Categorical:", len(CAT_COLS))


## 3. Load and quality-filter training data

Only records with accepted winter and summer processing methods (`1` or `2`) are retained, matching the final training workflow.


In [ ]:
if not TRAIN_PATH.exists():
    raise FileNotFoundError(
        f"Training dataset not found:\n  {TRAIN_PATH}\n\n"
        "Place the prepared training GeoParquet at data/training_roofs.parquet "
        "or update TRAIN_PATH above. See data/README.md for the expected schema."
    )

gdf_train = gpd.read_parquet(TRAIN_PATH)

required = set(
    FEATURES_FINAL
    + [TARGET_COL, GROUP_COL, "method_winter", "method_summer"]
)

missing = sorted(required - set(gdf_train.columns))
if missing:
    raise KeyError(f"Training dataset is missing required columns: {missing}")

mask = (
    gdf_train["method_winter"].isin(ALLOWED_METHODS)
    & gdf_train["method_summer"].isin(ALLOWED_METHODS)
)
gdf_train = gdf_train.loc[mask].copy()

X_all = gdf_train[FEATURES_FINAL].copy()
y_all = gdf_train[TARGET_COL].astype(int).to_numpy()
groups_city = gdf_train[GROUP_COL].astype(str).to_numpy()

print("Training rows:", len(gdf_train))
print("\nLabel counts:")
print(pd.Series(y_all).value_counts().sort_index())
print("\nRows by city:")
print(pd.Series(groups_city).value_counts())


## 4. Preprocessing and Random Forest pipeline

Numeric features are median-imputed with missing-value indicators. Categorical features are imputed with the most frequent value and one-hot encoded. The full preprocessing stage is stored together with the classifier in a single scikit-learn `Pipeline`.


In [ ]:
def make_preprocessor():
    numeric = SimpleImputer(
        strategy="median",
        add_indicator=True,
    )

    categorical = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("ohe", OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=True,
        )),
    ])

    return ColumnTransformer(
        [
            ("num", numeric, NUM_COLS),
            ("cat", categorical, CAT_COLS),
        ],
        remainder="drop",
        sparse_threshold=0.3,
    )


def make_rf_pipeline(random_state=42):
    rf = RandomForestClassifier(
        n_estimators=600,
        min_samples_leaf=2,
        n_jobs=-1,
        random_state=random_state,
        class_weight="balanced_subsample",
    )

    return Pipeline([
        ("prep", make_preprocessor()),
        ("clf", rf),
    ])


pipe_rf = make_rf_pipeline()
pipe_rf


## 5. Evaluation helpers


In [ ]:
def best_f1_from_pr(y_true, prob):
    precision, recall, thresholds = precision_recall_curve(y_true, prob)

    if len(thresholds) == 0:
        return {
            "threshold": 0.5,
            "f1": np.nan,
            "precision": np.nan,
            "recall": np.nan,
        }

    p = precision[:-1]
    r = recall[:-1]
    f1 = (2 * p * r) / (p + r + 1e-9)

    i = int(np.argmax(f1))

    return {
        "threshold": float(thresholds[i]),
        "f1": float(f1[i]),
        "precision": float(p[i]),
        "recall": float(r[i]),
    }


def classification_at_threshold(y_true, prob, threshold):
    y_hat = (np.asarray(prob) >= threshold).astype(int)

    return {
        "accuracy": float(accuracy_score(y_true, y_hat)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_hat)),
        "f1": float(f1_score(y_true, y_hat, zero_division=0)),
        "selected": int(y_hat.sum()),
        "selection_rate": float(y_hat.mean()),
    }


## 6. Leave-One-City-Out cross-validation

LOCO holds out one complete city at a time. This is the more demanding evaluation because the model is tested on a geographically unseen group rather than on randomly mixed roof samples.


In [ ]:
logo = LeaveOneGroupOut()
rows = []

for fold_i, (tr_idx, te_idx) in enumerate(
    logo.split(X_all, y_all, groups=groups_city),
    start=1,
):
    X_tr = X_all.iloc[tr_idx]
    X_te = X_all.iloc[te_idx]

    y_tr = y_all[tr_idx]
    y_te = y_all[te_idx]

    test_city = pd.Series(groups_city[te_idx]).unique()[0]

    pipe = make_rf_pipeline()
    sample_weight = compute_sample_weight(
        class_weight="balanced",
        y=y_tr,
    )

    t0 = time.time()
    pipe.fit(
        X_tr,
        y_tr,
        clf__sample_weight=sample_weight,
    )
    fit_seconds = time.time() - t0

    prob = pipe.predict_proba(X_te)[:, 1]

    ap = float(average_precision_score(y_te, prob))
    best = best_f1_from_pr(y_te, prob)
    bacc = balanced_accuracy_score(
        y_te,
        (prob >= best["threshold"]).astype(int),
    )

    rows.append({
        "fold": fold_i,
        "test_city": test_city,
        "n_train": len(tr_idx),
        "n_test": len(te_idx),
        "positive_test": int(y_te.sum()),
        "pr_auc": ap,
        "best_f1": best["f1"],
        "precision_best_f1": best["precision"],
        "recall_best_f1": best["recall"],
        "balanced_accuracy_best_f1": float(bacc),
        "fit_seconds": fit_seconds,
    })

    print(
        f"Fold {fold_i:02d} | {test_city} | "
        f"PR-AUC={ap:.4f} | "
        f"F1={best['f1']:.4f} | "
        f"fit={fit_seconds:.1f}s"
    )

df_loco = pd.DataFrame(rows)
df_loco


In [ ]:
loco_summary = (
    df_loco[
        [
            "pr_auc",
            "best_f1",
            "precision_best_f1",
            "recall_best_f1",
            "balanced_accuracy_best_f1",
        ]
    ]
    .agg(["mean", "std"])
    .round(4)
)

print("LOCO summary")
display(loco_summary)


## 7. Repeated stratified mixed-sample cross-validation

This second evaluation ignores city grouping and uses repeated stratified random folds. It is useful as a conventional ML baseline, but it should not be interpreted as a substitute for geographic hold-out validation.


In [ ]:
rskf = RepeatedStratifiedKFold(
    n_splits=5,
    n_repeats=3,
    random_state=42,
)

rows = []

for fold_i, (tr_idx, te_idx) in enumerate(
    rskf.split(X_all, y_all),
    start=1,
):
    X_tr = X_all.iloc[tr_idx]
    X_te = X_all.iloc[te_idx]

    y_tr = y_all[tr_idx]
    y_te = y_all[te_idx]

    pipe = make_rf_pipeline()
    sample_weight = compute_sample_weight(
        class_weight="balanced",
        y=y_tr,
    )

    pipe.fit(
        X_tr,
        y_tr,
        clf__sample_weight=sample_weight,
    )

    prob = pipe.predict_proba(X_te)[:, 1]

    ap = float(average_precision_score(y_te, prob))
    best = best_f1_from_pr(y_te, prob)

    rows.append({
        "fold": fold_i,
        "pr_auc": ap,
        "best_f1": best["f1"],
        "precision_best_f1": best["precision"],
        "recall_best_f1": best["recall"],
    })

df_mixed = pd.DataFrame(rows)

mixed_summary = (
    df_mixed[
        [
            "pr_auc",
            "best_f1",
            "precision_best_f1",
            "recall_best_f1",
        ]
    ]
    .agg(["mean", "std"])
    .round(4)
)

print("Repeated stratified CV summary")
display(mixed_summary)


## 8. Validation-selected threshold and held-out test evaluation

A stratified 60/20/20 split is used here only to demonstrate threshold selection. The decision threshold is selected from the validation set and then applied unchanged to the held-out test set.


In [ ]:
X_train, X_tmp, y_train, y_tmp = train_test_split(
    X_all,
    y_all,
    test_size=0.40,
    stratify=y_all,
    random_state=42,
)

X_val, X_test, y_val, y_test = train_test_split(
    X_tmp,
    y_tmp,
    test_size=0.50,
    stratify=y_tmp,
    random_state=42,
)

pipe_split = make_rf_pipeline()

sample_weight = compute_sample_weight(
    class_weight="balanced",
    y=y_train,
)

pipe_split.fit(
    X_train,
    y_train,
    clf__sample_weight=sample_weight,
)

p_val = pipe_split.predict_proba(X_val)[:, 1]
p_test = pipe_split.predict_proba(X_test)[:, 1]

best_val = best_f1_from_pr(y_val, p_val)
threshold = best_val["threshold"]

test_pr_auc = float(
    average_precision_score(y_test, p_test)
)

test_metrics = classification_at_threshold(
    y_test,
    p_test,
    threshold,
)

print("Validation-selected threshold:", round(threshold, 4))
print("Test PR-AUC:", round(test_pr_auc, 4))
print("Test metrics:")
for key, value in test_metrics.items():
    print(f"  {key}: {value:.4f}" if isinstance(value, float) else f"  {key}: {value}")


In [ ]:
precision_val, recall_val, _ = precision_recall_curve(y_val, p_val)
precision_test, recall_test, _ = precision_recall_curve(y_test, p_test)

ap_val = average_precision_score(y_val, p_val)
ap_test = average_precision_score(y_test, p_test)

plt.figure(figsize=(7, 5))
plt.plot(recall_val, precision_val, label=f"Validation (AP={ap_val:.3f})")
plt.plot(recall_test, precision_test, label=f"Test (AP={ap_test:.3f})")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision–Recall Curves")
plt.legend()
plt.tight_layout()
plt.show()


## 9. Train and export the final production model

The final model is fitted on all available labelled training data. The exported `.joblib` contains both preprocessing and the Random Forest classifier.


In [ ]:
pipe_final = make_rf_pipeline()

sample_weight_full = compute_sample_weight(
    class_weight="balanced",
    y=y_all,
)

t0 = time.time()

pipe_final.fit(
    X_all,
    y_all,
    clf__sample_weight=sample_weight_full,
)

fit_seconds = time.time() - t0

joblib.dump(
    pipe_final,
    MODEL_PATH,
    compress=3,
)

model_size_mb = MODEL_PATH.stat().st_size / (1024 ** 2)

print(f"Final model trained in {fit_seconds:.1f} seconds")
print(f"Saved to: {MODEL_PATH}")
print(f"Model size: {model_size_mb:.1f} MB")


## 10. Reload and verify the exported model


In [ ]:
loaded_model = joblib.load(MODEL_PATH)

demo = X_all.head(5).copy()
demo_prob = loaded_model.predict_proba(demo)[:, 1]

demo_result = demo.copy()
demo_result["green_roof_probability"] = demo_prob

demo_result


## Notes for public use

- The model expects the exact 14 input features listed above.
- Categorical values unseen during training are handled by `OneHotEncoder(handle_unknown="ignore")`.
- The `.joblib` file uses Python pickle-based serialization. Only load model files from trusted sources.
- Geographic LOCO validation and randomly mixed cross-validation answer different questions; LOCO is the more relevant indicator of transfer to unseen cities.
- The full Baden-Württemberg prediction workflow is intentionally excluded from this publication notebook because it depends on large external datasets and project-specific processing infrastructure.
